In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append(".")
sys.path.append("../../")

import os
os.chdir("../..")

In [3]:
import numpy as np
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from tqdm import tqdm
import matplotlib.pyplot as plt

import copy

import hydra
import torch
import torch.nn.functional as F

from einops import einsum

from hydra import compose, initialize
from omegaconf import OmegaConf

from pathlib import Path

from anycam.common.array_operations import map_fn, unsqueezer
from anycam.visualization.common import color_tensor

from torch.cuda.amp import autocast
import math

import uuid

from torchvision.utils import flow_to_image

from anycam.datasets import make_datasets
from anycam.models import make_depth_predictor, make_pose_predictor
from anycam.trainer import AnyCamWrapper
from anycam.loss.metric import camera_to_rel_deg
from anycam.datasets.common import get_flow_selector, get_index_selector


from anycam.datasets.common import get_ids_for_sequence

from anycam.scripts.fit_video import fit_video

import rerun as rr
import rerun.blueprint as rrb


from minipytorch3d.rotation_conversions import (
    matrix_to_quaternion,
    quaternion_to_matrix,
    matrix_to_axis_angle,
    axis_angle_to_matrix,
)

E:\AnyCam\anycam\env\lib\site-packages\ignite\handlers\checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [4]:
def filter_depth(depth, threshold=0.1):
    _, h, w = depth.shape

    depth = depth.clone()[None, ...]
    median = torch.median(depth)
    
    depth_grad = torch.stack(torch.gradient(depth, dim=(-2, -1))).norm(dim=0)

    mask = depth_grad < median * threshold

    return mask


def visualize_rerun(trajectory, depths, imgs, proj, uncertainties=None, cam_every=1, depth_every=1, subsample_pts=16, step=0, uncertainty_thresh=-1, radii=1, track_masks=None, max_depth_masks=None, filter_depth_threshold=0.1, static_points_accumulate=False, highlight_dynamic=False, image_plane_distance=0.05):
    h, w = imgs[0].shape[:2]

    rr.set_time_sequence("step", step)

    if cam_every > 0:
        for i, d_id in enumerate(list(range(len(depths)))[cam_every//2::cam_every]):
            if static_points_accumulate:
                rr.set_time_sequence("step", d_id)

            pose = trajectory[d_id]
            rr.log(f"world/scene/cameras/{i:03d}", rr.Pinhole(
                resolution=[w, h],
                focal_length=float(proj[0, 0]),
                image_plane_distance=image_plane_distance, 
            ), static=not static_points_accumulate)
            rr.log(f"world/scene/cameras/{i:03d}", rr.Transform3D(translation=pose[:3, 3].cpu(), mat3x3=pose[:3, :3].cpu(), axis_length=0.01), static=not static_points_accumulate)

            depth_img = color_tensor(depths[d_id], cmap="plasma")[0]
            depth_img = depth_img.cpu().numpy()
            depth_img = (depth_img * 255).astype(np.uint8)

            # rr.log(f"world/scene/cameras/{i:03d}/depth", rr.Image(depth_img).compress())

            if uncertainties is not None:
                uncertainty_img = color_tensor(uncertainties[d_id].clamp(max=torch.quantile(uncertainties[d_id], .95)), cmap="plasma", norm=True)[0]
                uncertainty_img = uncertainty_img.cpu().numpy()
                uncertainty_img = (uncertainty_img * 255).astype(np.uint8)

                # rr.log(f"world/scene/cameras/{i:03d}/uncertainty", rr.Image(uncertainty_img).compress())

            rr.log(f"world/scene/cameras/{i:03d}/input", rr.Image((imgs[d_id] * 255).astype(np.uint8)).compress(jpeg_quality=95), static=not static_points_accumulate)

            if highlight_dynamic > 0 and (d_id - cam_every//2) % cam_every == 0:
                pts, colors = lift_image(torch.tensor(imgs[d_id]).cuda(), depths[d_id].cuda(), trajectory[d_id].cuda(), proj)
                dyn_mask = uncertainties[d_id].view(-1) > highlight_dynamic
                dyn_mask = dyn_mask & filter_depth(depths[d_id].cuda(), threshold=filter_depth_threshold).view(-1)

                pts_dyn = pts[dyn_mask, :]
                colors_dyn = colors[dyn_mask, :]

                colors_dyn = colors_dyn * .5 + colors_dyn.new_tensor((1, 0, 0)) * .5

                colors_dyn = (colors_dyn * 255).clamp(0, 255).to(torch.uint8)

                rr.log(f"world/scene/points/{i:03d}_dyn", rr.Points3D(pts_dyn[:, :3].cpu().numpy(), colors=colors_dyn[:, :3].cpu().numpy(), radii=rr.Radius.ui_points([radii]),), static=not static_points_accumulate)


    xyz_min, xyz_max = None, None

    for i, t_id in enumerate(list(range(len(depths)))[depth_every//2::depth_every]):
        pts, colors = lift_image(torch.tensor(imgs[t_id]).cuda(), depths[t_id].cuda(), trajectory[t_id].cuda(), proj)
        mask = filter_depth(depths[t_id].cuda(), threshold=filter_depth_threshold)

        mask = mask.view(-1)

        mask = mask & (depths[t_id].cuda().view(-1) < 10)

        if uncertainty_thresh > 0:
            mask = mask & (uncertainties[t_id].view(-1) < uncertainty_thresh)

        if track_masks is not None:
            mask = mask & track_masks[t_id].view(-1)

        if max_depth_masks is not None:
            mask = mask & max_depth_masks[t_id].view(-1)           

        pts_filtered = pts[mask, :]
        colors_filtered = colors[mask, :]

        pts_filtered = pts_filtered[subsample_pts//2::subsample_pts]
        colors_filtered = colors_filtered[subsample_pts//2::subsample_pts]

        if static_points_accumulate:
            rr.set_time_sequence("step", t_id)

        colors_filtered = (colors_filtered * 255).clamp(0, 255).to(torch.uint8)

        rr.log(f"world/scene/points/{i:03d}", rr.Points3D(pts_filtered[:, :3].cpu().numpy(), colors=colors_filtered[:, :3].cpu().numpy(), radii=rr.Radius.ui_points([radii]),), static=not static_points_accumulate)
    

        if xyz_min is None and pts_filtered.shape[0] > 0:
            xyz_min = pts_filtered.min(dim=0).values
            xyz_max = pts_filtered.max(dim=0).values
        elif pts_filtered.shape[0] > 0:
            xyz_min = torch.min(xyz_min, pts_filtered.min(dim=0).values)
            xyz_max = torch.max(xyz_max, pts_filtered.max(dim=0).values)

    return xyz_min, xyz_max


def visualize_video_rerun(
        trajectory, 
        depths, 
        imgs, 
        proj,
        uncertainties=None, 
        cam_every=-1, 
        depth_every=1, 
        subsample_pts=1, 
        subsample_dynamic_pts=1, 
        step=0, 
        radii=1.5, 
        uncertainty_thresh=-1, 
        track_masks=None, 
        max_depth_masks=None, 
        follow_cam=None, 
        filter_depth_threshold=0.1,
        static_points_accumulate=False,
        highlight_dynamic=False,
        image_plane_distance=0.05,
        ):
    h, w = imgs[0].shape[:2]

    imgs = np.array(imgs)

    # imgs_grey = (imgs.astype(float) * .5 + .5)
    imgs_grey = imgs

    xyz_min, xyz_max = visualize_rerun(
        trajectory, 
        depths, 
        imgs_grey, 
        proj, 
        uncertainties, 
        cam_every=cam_every, 
        depth_every=depth_every, 
        subsample_pts=subsample_pts, 
        step=step, 
        uncertainty_thresh=uncertainty_thresh, 
        radii=radii, 
        track_masks=track_masks, 
        max_depth_masks=max_depth_masks, 
        filter_depth_threshold=filter_depth_threshold,
        static_points_accumulate=static_points_accumulate,
        highlight_dynamic=highlight_dynamic,
        image_plane_distance=image_plane_distance,
        )

    for t_id in range(len(depths)):
        rr.set_time_sequence("step", t_id+step)

        pts, colors = lift_image(torch.tensor(imgs[t_id]).cuda(), depths[t_id].cuda() - 0.01, trajectory[t_id].cuda(), proj)
        mask = filter_depth(depths[t_id].cuda(), threshold=filter_depth_threshold)

        mask = mask.view(-1)

        mask = mask & (uncertainties[t_id].view(-1) >= uncertainty_thresh)

        if max_depth_masks is not None:
            mask = mask & max_depth_masks[t_id].view(-1)

        pts = pts[mask, :]
        colors = colors[mask, :]

        pts = pts[subsample_dynamic_pts//2::subsample_dynamic_pts]
        colors = colors[subsample_dynamic_pts//2::subsample_dynamic_pts]
        colors = (colors * 255).clamp(0, 255).to(torch.uint8)

        if xyz_min is None and pts.shape[0] > 0:
            xyz_min = pts.min(dim=0).values
            xyz_max = pts.max(dim=0).values
        elif pts.shape[0] > 0:
            xyz_min = torch.min(xyz_min, pts.min(dim=0).values)
            xyz_max = torch.max(xyz_max, pts.max(dim=0).values)

        rr.log(f"world/scene/active_points", rr.Points3D(pts[:, :3].cpu().numpy(), colors=colors[:, :3].cpu().numpy(), radii=rr.Radius.ui_points([radii]),))

        pose = trajectory[t_id]
        rot = quaternion_to_matrix(matrix_to_quaternion(pose[:3, :3].cpu()))

        rr.log(f"world/scene/active_cam", rr.Pinhole(
            resolution=[w, h],
            focal_length=float(proj[0, 0]),
            image_plane_distance=image_plane_distance, 
        ), static=True)
        rr.log(f"world/scene/active_cam", rr.Transform3D(translation=pose[:3, 3].cpu(), mat3x3=rot, axis_length=0.01))

        rr.log("world/scene/cam_traj", rr.LineStrips3D([pose[:3, 3].cpu().numpy().tolist() for pose in trajectory[:t_id+1]], colors=[(0, 255, 0)]), static=False)

        rr.log("world/scene/active_cam/input", rr.Image((imgs[t_id] * 255).astype(np.uint8)).compress(jpeg_quality=95))
        
        if uncertainties is not None:
            uncertainty_img = color_tensor((uncertainties[t_id] / 0.05).clamp(0, 1), cmap="plasma", norm=False)[0]
            uncertainty_img = uncertainty_img.cpu().numpy()
            uncertainty_img = (uncertainty_img * 255).astype(np.uint8)

            rr.log(f"world/scene/active_cam/uncertainty", rr.Image(uncertainty_img).compress())


        if follow_cam == "world":
            rr.log("world/scene", rr.Transform3D(translation=pose[:3, 3].cpu(), mat3x3=rot, axis_length=0, from_parent=True))

        elif follow_cam is not None:
            fc_t, fc_a = follow_cam
            rr.log(f"world/scene/ac/follow_cam", rr.Pinhole(
                resolution=[1280, 720],
                focal_length=1280,
                image_plane_distance=0.02, 
            ), static=True)
            rr.log(f"world/scene/ac", rr.Transform3D(translation=pose[:3, 3].cpu(), mat3x3=rot, axis_length=0.01))
            rr.log(f"world/scene/ac/follow_cam", rr.Transform3D(translation=fc_t, rotation=rr.RotationAxisAngle(axis=[0, 1, 0], angle=rr.Angle(deg=fc_a)) ,axis_length=0.01))

    xyz_abs_max = torch.max(torch.abs(xyz_min), torch.abs(xyz_max)).cpu()
    bbox_pts = torch.tensor([
        [-1, -1, -1],
        [1, -1, -1],
        [1, 1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
        [1, -1, 1],
        [1, 1, 1],
        [-1, 1, 1],
    ]) * xyz_abs_max

    # rr.log("world/scene/bbox", rr.Points3D(bbox_pts.numpy(), colors=[(255, 255, 255)]), static=True)

In [5]:
from anycam.common.geometry import get_grid_xy


def get_normals(depth, proj):
    # Get grid

    n, _, h, w = depth.shape

    xy = get_grid_xy(h, w, homogeneous=True, device=depth.device)

    # Project to 3D
    pts = torch.inverse(proj) @ xy.reshape(3, -1)

    pts = pts * depth.view(n, 1, -1)

    # Get normals
    pts = pts.reshape(n, 3, h, w)

    dx, dy = torch.gradient(pts, dim=(-2, -1))

    n = torch.cross(dx, dy, dim=1)

    n = n / n.norm(dim=1, keepdim=True)

    return n


def lift_image(img, depth, pose, proj):
    h, w = img.shape[:2]

    proj = torch.tensor(proj, device=device).float()

    proj[0, 0] = proj[0, 0] / w * 2
    proj[1, 1] = proj[1, 1] / h * 2
    proj[0, 2] = proj[0, 2] / w * 2 - 1
    proj[1, 2] = proj[1, 2] / h * 2 - 1

    inv_proj = torch.inverse(proj)

    pts = get_grid_xy(h, w, homogeneous=True).reshape(3, h*w).cuda()
    pts = inv_proj @ pts
    pts = pts * depth.view(1, -1).cuda()
    pts = torch.cat((pts, torch.ones(1, h*w, device=device)), dim=0)
    pts = pose @ pts
    pts = pts[:3, :].T

    colors = torch.tensor(img.reshape(-1, 3)).cuda()

    return pts, colors

In [6]:
def get_depth_flow(imgs, model):
    depths = []
    flow_occs_fwd = []
    flow_occs_bwd = []

    c, h, w = imgs.shape[1:]

    with torch.no_grad():
        for i in tqdm(range(len(imgs)-1)):
            img0 = imgs[i]
            img1 = imgs[i+1]

            imgs_ = torch.stack([img0, img1]).unsqueeze(0).cuda()

            images_ip_fwd, images_ip_bwd = model.image_processor(imgs_ * 2 - 1, data={})

            flow_occs_fwd.append(images_ip_fwd[0, :(1 if i != len(imgs)-2 else 2), 3:6].cpu())
            flow_occs_bwd.append(images_ip_bwd[0, (1 if i != 0 else 0):, 3:6].cpu())

            depth = model.depth_predictor(img0.unsqueeze(0).cuda())

            depth = 1 / depth[0].clamp_min(1e-3)

            depths.append(depth.cpu())

        depth = model.depth_predictor(imgs[-1].unsqueeze(0).cuda())
        depth = 1 / depth[0].clamp_min(1e-3)

        depths.append(depth.cpu())

    seq_depths = torch.cat(depths, dim=0)
    seq_flow_occs_fwd = torch.cat(flow_occs_fwd, dim=0)
    seq_flow_occs_bwd = torch.cat(flow_occs_bwd, dim=0)

    return seq_depths, seq_flow_occs_fwd, seq_flow_occs_bwd

In [7]:
from depthcrafter.depth_crafter_ppl import DepthCrafterPipeline
from depthcrafter.unet import DiffusersUNetSpatioTemporalConditionModelDepthCrafter


def get_depth_crafter():
    unet = DiffusersUNetSpatioTemporalConditionModelDepthCrafter.from_pretrained(
        "tencent/DepthCrafter",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    pipe = DepthCrafterPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt",
        unet=unet,
        torch_dtype=torch.float16,
        variant="fp16",
    )
    pipe.to("cuda")
    return pipe

def depth_crafter_inference(pipe, imgs):
    # imgs = imgs.permute(0, 2, 3, 1).float().cuda()
    n, h, w, c = imgs.shape

    imgs = imgs.permute(0, 3, 1, 2)

    th = math.ceil(h / 128) * 128
    tw = math.ceil(w / 128) * 128

    imgs = F.interpolate(imgs, (th, tw))

    imgs = imgs.permute(0, 2, 3, 1).cpu().numpy()

    res = pipe(
            imgs,
            height=imgs.shape[1],
            width=imgs.shape[2],
            output_type="np",
            guidance_scale=1,
            num_inference_steps=5,
            window_size=110,
            overlap=25,
            track_time=True,
        ).frames[0]

    res = res.sum(-1) / res.shape[-1]

    res = F.interpolate(torch.tensor(res, device="cuda").unsqueeze(1), (h, w)).cpu()

    res = 1 / res

    return res

A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "E:\AnyCam\anycam\env\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'


In [8]:
def depth_alignment(depths_uni, depths_dc, uncertainties):
    n, _, h, w = depths_uni.shape

    depths_uni_opt = depths_uni[:-1].reshape(-1).cuda()
    depths_dc_opt = depths_dc[:-1].reshape(-1).cuda()
    uncertainties = uncertainties.reshape(-1)

    mask = (depths_uni_opt > 0) & (depths_uni_opt < 50)

    depths_uni_opt = depths_uni_opt[mask]
    depths_dc_opt = depths_dc_opt[mask]
    uncertainties = uncertainties[mask]

    depths_uni_opt = 1 / depths_uni_opt
    depths_dc_opt = 1 / depths_dc_opt

    A = torch.stack([depths_dc_opt, torch.ones_like(depths_dc_opt)], dim=1)
    b = depths_uni_opt.view(-1, 1)

    weight = 1 / uncertainties.clamp_min(1e-3)
    weight = weight.view(-1, 1)

    A = A * weight
    b = b * weight

    x = torch.linalg.lstsq(A, b).solution

    scale = x[0]
    shift = x[1]

    print(f"Aligning with scale {scale.item()} and shift {shift.item()}")

    depths_dc_aligned = 1 / ((1 / depths_dc.cuda()) * scale + shift).clamp_min(1e-3)

    return depths_dc_aligned.cpu()

In [9]:
def compute_track_masks(flow_occs, uncertainties, max_track_len=8, uncertainty_thresh=0.1, min_track_len=1, plot_rerun=False, imgs=None):
    n, _, h, w = flow_occs.shape
    device = flow_occs.device

    masks = torch.zeros_like(flow_occs[:, :1, :, :])
    tracks = torch.zeros((0, 2), device=device, dtype=torch.long)
    track_lens = torch.zeros((0,), device=device, dtype=torch.long)
    track_starts = torch.zeros((0, 3), device=device, dtype=torch.float)

    
    if plot_rerun:
        rr.init("Track Masks", recording_id=uuid.uuid4())
        rr.connect()

    for i_curr_frame in range(n):
        if plot_rerun:
            rr.set_time_sequence("step", i_curr_frame)
            rr.log("2d/img", rr.Image((imgs[i_curr_frame] * 255).astype(np.uint8)).compress())

        # If maximum track length has been reached, clear the track
        len_mask = track_lens < max_track_len

        tracks = tracks[len_mask]
        track_lens = track_lens[len_mask]
        track_starts = track_starts[len_mask]


        # Create mask of pixels that are currently being tracked

        tracked_pixels = torch.zeros_like(masks[0, 0], dtype=torch.bool)

        tracked_pixels[tracks[:, 1], tracks[:, 0]] = True

        
        # Add new tracks for untracked pixels

        x = torch.arange(w, device=device).view(1, -1).expand(h, -1)
        y = torch.arange(h, device=device).view(-1, 1).expand(-1, w)
        xy = torch.stack([x, y], dim=-1).view(-1, 2)

        new_tracks = xy[~tracked_pixels.view(-1), :]
        new_track_lens = torch.ones_like(new_tracks[:, 0])
        new_track_starts = torch.cat([new_tracks, i_curr_frame * torch.ones_like(new_tracks[:, 0]).view(-1, 1)], dim=1)


        # Filter both tracks and new tracks by uncertainty

        track_uncs = uncertainties[i_curr_frame, 0, tracks[:, 1], tracks[:, 0]]
        new_track_uncs = uncertainties[i_curr_frame, 0, new_tracks[:, 1], new_tracks[:, 0]]

        tracks = tracks[track_uncs < uncertainty_thresh]
        track_lens = track_lens[track_uncs < uncertainty_thresh]
        track_starts = track_starts[track_uncs < uncertainty_thresh]

        new_tracks = new_tracks[new_track_uncs < uncertainty_thresh]
        new_track_lens = new_track_lens[new_track_uncs < uncertainty_thresh]
        new_track_starts = new_track_starts[new_track_uncs < uncertainty_thresh]

        if plot_rerun:
            rr.log("2d/tracks", rr.Points2D(tracks.cpu().numpy(), colors=[(255, 255, 0)]))
            rr.log("2d/new_tracks", rr.Points2D(new_tracks.cpu().numpy(), colors=[(0, 255, 0)]))


        # Add new tracks to track list
        tracks = torch.cat([tracks, new_tracks], dim=0)
        track_lens = torch.cat([track_lens, new_track_lens], dim=0)
        track_starts = torch.cat([track_starts, new_track_starts], dim=0)


        # Register new tracks in mask

        # masks[i_curr_frame, 0, new_tracks[:, 1], new_tracks[:, 0]] = True
        
        register_mask = track_lens == min_track_len

        masks[i_curr_frame, 0, tracks[register_mask, 1], tracks[register_mask, 0]] = True


        # Check if track is occluded. If so, remove it

        track_occs = flow_occs[i_curr_frame, 2, :, :][tracks[:, 1], tracks[:, 0]] > .5


        if plot_rerun:
            rr.log("2d/occluded_tracks", rr.Points2D(tracks[~track_occs].cpu().numpy(), colors=[(255, 0, 0)]))


        tracks = tracks[track_occs]
        track_lens = track_lens[track_occs]
        track_starts = track_starts[track_occs]


        # Compute flow for tracked pixels

        track_flow_x = flow_occs[i_curr_frame, 0, :, :][tracks[:, 1], tracks[:, 0]]
        track_flow_y = flow_occs[i_curr_frame, 1, :, :][tracks[:, 1], tracks[:, 0]]

        track_flow_x = track_flow_x * w
        track_flow_y = track_flow_y * h

        track_flow = torch.stack([track_flow_x, track_flow_y], dim=-1)

        tracks = tracks.float() + track_flow
        tracks = tracks.round().long()


        # Check if track is out of bounds. If so, remove it

        out_of_bounds = (tracks[:, 0] < 0) | (tracks[:, 0] >= w) | (tracks[:, 1] < 0) | (tracks[:, 1] >= h)

        tracks = tracks[~out_of_bounds]
        track_lens = track_lens[~out_of_bounds]
        track_starts = track_starts[~out_of_bounds]

        
        # Update track_lens

        track_lens += 1


    return masks

In [10]:
from anycam.datasets.davis.davis_dataset import DavisDataset


davis = DavisDataset(
    data_path="E:\AnyCam\my_data",
    split_path=None,
    image_size=[336, int(16/9*336)],
    # image_size=336,
    frame_count=2,
    return_depth=False,
    return_flow=False,
    dilation=1,
    index_selector=get_index_selector(True),
    flow_selector=get_flow_selector(1, True)
)

In [13]:
from anycam.datasets.common import get_sequence_sampler
from anycam.datasets.waymo.waymo_dataset import WaymoDataset

waymo =  WaymoDataset(
    data_path="data/waymo/testing",
    split_path="anycam/datasets/waymo/splits/eval_seqs_2_64/test_files.txt",
    image_size=[336, int(16/9*336)],
    frame_count=2,
    return_depth=False,
    return_flow=False,
    index_selector=get_index_selector(True),
    sequence_sampler=get_sequence_sampler(True),
    flow_selector=get_flow_selector(0, True),
)

FileNotFoundError: [WinError 3] 系统找不到指定的路径。: 'data\\waymo\\testing'

In [14]:
from anycam.datasets.aria_everyday_activities.aea_dataset import ExtractedAEADataset
from anycam.datasets.common import get_index_selector


ext_aea = ExtractedAEADataset(
    data_path="data/aea_extracted",
    split_path=None,
    image_size=336,
    frame_count=3,
    return_depth=False,
    return_flow=False,
    dilation=1,
    index_selector=get_index_selector(True),
    flow_selector=get_flow_selector(1, True),
    preprocessed_path="/storage/slurm/wimbauer/unimatch_flows/aea/",
)

FileNotFoundError: [WinError 3] 系统找不到指定的路径。: 'data/aea_extracted'

In [ ]:
ext_aea = ExtractedAEADataset(
    data_path="anycam/aea_extracted",
    split_path=None,
    image_size=336,
    frame_count=3,
    return_depth=False,
    return_flow=False,
    dilation=1,
    index_selector=get_index_selector(True),
    flow_selector=get_flow_selector(1, True),
    # preprocessed_path="data/unimatch_flows/aea/",
)

100%|██████████| 30/30 [00:00<00:00, 102801.57it/s]


In [11]:
import cv2

def load_and_resize_frames(folder_path, target_size=336):
    frames = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(folder_path, filename)
            frame = cv2.imread(file_path)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
           
            height, width = frame.shape[:2]
            if height < width:
                new_height = target_size
                new_width = int((target_size / height) * width)
            else:
                new_width = target_size
                new_height = int((target_size / width) * height)
            
            resized_frame = cv2.resize(frame, (new_width, new_height))

            resized_frame = resized_frame.astype(np.float32) / 255.0

            frames.append(resized_frame)
    
    return frames

folder_path = 'custom_data/running_dog_pexels'
imgs = load_and_resize_frames(folder_path)

FileNotFoundError: [WinError 3] 系统找不到指定的路径。: 'custom_data/running_dog_pexels'

In [12]:
davis._sequences

{'my_video': 249}

In [13]:
from anycam.scripts.common import load_model
from anycam.loss import make_loss
from anycam.trainer import AnyCamWrapper

def load_anycam(model_path, checkpoint=None):
    with initialize(version_base=None, config_path=str("../.." / model_path), job_name="test_app"):
        config = compose(config_name="training_config")

    prefix = "training_checkpoint_"
    ckpts = Path(model_path).glob(f"{prefix}*.pt")

    model_conf = config["model"]
    model_conf["use_provided_flow"] = False
    model_conf["train_directions"] = "forward"

    model = AnyCamWrapper(model_conf)

    criterion = [make_loss(cfg) for cfg in config.get("loss", [])][0]

    training_steps = [int(ckpt.stem.split(prefix)[1]) for ckpt in ckpts]

    print(training_steps)
    if training_steps:
        if checkpoint is None:
            ckpt_path = f"{prefix}{max(training_steps)}.pt"
        else:
            ckpt_path = checkpoint

        # ckpt_path = Path(config["output"]["path"]) / ckpt_path
        ckpt_path = Path(model_path) / ckpt_path

        print(ckpt_path)

        cp = torch.load(ckpt_path, map_location="cpu")

        cp["model"] = {k: v for k, v in cp["model"].items() if not "depth_predictor" in k}

        model.load_state_dict(cp["model"], strict=False)

    return model, criterion


run_name = "anycam_seq8"

# model_path = Path("out/vacation_runs") / run_name
model_path = Path("pretrained_models") / run_name

device = "cuda"

dataset = davis
sequence = "my_video"

# ids = list(range(515, 575))
# ids = list(range(896, 960)) + list(range(1920, 1984))
ids, _ = get_ids_for_sequence(dataset, sequence)
# ids = [i for i, dp in enumerate(dataset._datapoints) if dataset._datapoints[0][0] == dp[0]]

# ids = ids[80:280]

print(f"Found {len(ids)} datapoints")

# imgs = []
# poses = []
# proj = None

# gt_proj = None

# for id in ids:
#     data = dataset[id]
#     imgs.append(data["imgs"][0].transpose(1, 2, 0))
#     poses.append(data["poses"][0])
#     proj = data["projs"][0]

out_path = Path("media") / "anycam" / f"{dataset.NAME.strip()}_{run_name}"
out_path.mkdir(exist_ok=True, parents=True)

Found 248 datapoints


In [14]:
import os

# 为 Windows 系统伪造一个 HOME 环境变量，指向你的用户目录
if "HOME" not in os.environ:
    os.environ["HOME"] = os.environ.get("USERPROFILE", "")
    print(f"✅ 已设置 HOME 环境变量为: {os.environ['HOME']}")

✅ 已设置 HOME 环境变量为: C:\Users\Lenovo


In [15]:
model, criterion = load_anycam(model_path)
model = model.to(device)

Using cache found in C:\Users\Lenovo/.cache\torch\hub\Brummi_UniDepth_stable


UniDepth_v2old_vits14 is loaded with:
	 missing keys: []
	 additional keys: []


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[247500]
pretrained_models\anycam_seq8\training_checkpoint_247500.pt


In [16]:
import numpy as np
import torch

# 1. 检查是否已经获取了 ids（帧索引列表）
if 'ids' not in locals() or len(ids) == 0:
    # 如果 ids 没定义，尝试获取整个数据集的索引
    ids = list(range(len(dataset)))
    print(f"⚠️ ids 未定义，自动提取数据集全部 {len(ids)} 帧")

# 2. 提取图片数据
imgs_list = []
for i in ids:
    data = dataset[i]
    # AnyCam 数据集返回的通常是 [3, H, W]，且范围是 [0, 1] 的 Tensor
    # fit_video 函数通常需要 [H, W, 3] 的 numpy 数组
    img = data["imgs"][0] 
    if isinstance(img, torch.Tensor):
        img = img.permute(1, 2, 0).cpu().numpy()
    imgs_list.append(img)

# 3. 将列表转换为数组
imgs = np.stack(imgs_list)

print(f"✅ 成功准备 imgs 变量，形状为: {imgs.shape} (总帧数, 高, 宽, 通道)")

✅ 成功准备 imgs 变量，形状为: (248, 3, 336, 597) (总帧数, 高, 宽, 通道)


In [17]:
import numpy as np
import torch

# 确保 imgs 是 numpy 数组
if isinstance(imgs, list):
    imgs = np.stack(imgs)

# 打印一下形状，看看是不是 (数量, 高, 宽, 3)
print(f"当前 imgs 形状: {imgs.shape}")

# 如果形状是 (N, 3, H, W)，需要转回 (N, H, W, 3)
if imgs.shape[1] == 3 and imgs.shape[-1] != 3:
    imgs = imgs.transpose(0, 2, 3, 1)
    print(f"修正后形状: {imgs.shape}")

# 确保数据类型是 float32
if imgs.dtype != np.float32:
    imgs = imgs.astype(np.float32)

当前 imgs 形状: (248, 3, 336, 597)
修正后形状: (248, 336, 597, 3)


In [18]:
from dotdict import dotdict
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
fit_video_config = {
    "with_rerun": False,
    "do_ba_refinement": False,
    "overfit": False,
    "prediction": {
        "model_seq_len": 8,
        "shift": 7,
        # "model_seq_len": 8,
        # "shift": 7,
        "square_crop": True,
        "return_all_uncerts": True,
    },
    "ba_refinement": {
        "with_rerun": True,
        "max_uncert": 0.05,
        "lambda_smoothness": 0.1,
        # "lambda_smoothness": 0.001,
        "long_tracks": True,
        "n_steps_last_global": 1000,
    },
    "ba_refinement_level": 1,
    # "ba_refinement_level": 0,
    "dataset": {
        "image_size": [192, 336]
        # "image_size": [None, 336],
    }
}

fit_video_config = dotdict(fit_video_config)

best_trajectory, proj, extras_dict, ba_extras = fit_video(
    fit_video_config,
    model,
    criterion,
    imgs[::8],
    return_extras=True,
)

{'with_rerun': False, 'do_ba_refinement': False, 'overfit': False, 'prediction': {'model_seq_len': 8, 'shift': 7, 'square_crop': True, 'return_all_uncerts': True}, 'ba_refinement': {'with_rerun': True, 'max_uncert': 0.05, 'lambda_smoothness': 0.1, 'long_tracks': True, 'n_steps_last_global': 1000}, 'ba_refinement_level': 1, 'dataset': {'image_size': [192, 336]}}
dataset_config: {'image_size': [192, 336]}
do_ba_refinement: False
ba_refinement_level: 2
ba_refinement_config: {'with_rerun': True, 'max_uncert': 0.05, 'lambda_smoothness': 0.1, 'long_tracks': True, 'n_steps_last_global': 1000}
prediction_config: {'model_seq_len': 8, 'shift': 7, 'square_crop': True, 'return_all_uncerts': True}
model_seq_len: 8
shift: 7
proj_strategy: weighted
proj_label_source: prediction


E:\AnyCam\anycam\env\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  7.34it/s]

Best candidate: 31


In [19]:
best_trajectory = torch.tensor(best_trajectory)


proj_rel = torch.tensor(proj).clone()

h, w = imgs[0].shape[:2]

proj_rel[0, 0] = proj_rel[0, 0] / w * 2
proj_rel[0, 2] = proj_rel[0, 2] / w * 2 - 1
proj_rel[1, 1] = proj_rel[1, 1] / h * 2
proj_rel[1, 2] = proj_rel[1, 2] / h * 2 - 1

proj = torch.tensor(proj)
proj_rel = torch.tensor(proj_rel).cuda()

best_candidate = extras_dict["best_candidate"]

depths = extras_dict["seq_depths"]
#uncertainties = torch.stack(extras_dict["uncertainties"])[:, 0, best_candidate, :1, :, :]
uncertainties = torch.stack([u.cpu() for u in extras_dict["uncertainties"]])[:, 0, best_candidate, :1, :, :]
#flows_occs_fwd = extras_dict["seq_flow_occs_fwd"].cuda()
#flows_occs_bwd = extras_dict["seq_flow_occs_bwd"].cuda()
flows_occs_fwd = extras_dict["seq_flow_occs_fwd"].cpu()
flows_occs_bwd = extras_dict["seq_flow_occs_bwd"].cpu()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\1990169671.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  best_trajectory = torch.tensor(best_trajectory)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\1990169671.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  proj_rel = torch.tensor(proj_rel).cuda()


In [20]:
# Get UniDepth depths and flows

#depths_uni, seq_flows_occs_fwd, seq_flows_occs_bwd = get_depth_flow(torch.tensor(np.array(imgs), device="cuda").permute(0, 3, 1, 2), model)
import torch

# 1. 彻底清理显存残留
import gc
gc.collect()
torch.cuda.empty_cache()

# 2. 核心修改：先在 CPU 上创建 Tensor，不要直接加 device="cuda"
imgs_tensor = torch.as_tensor(np.array(imgs), dtype=torch.float32).permute(0, 3, 1, 2)

# 3. 尝试运行。如果 get_depth_flow 内部写得不够健壮（依然一次性搬运全量数据），
# 我们可能需要写个循环，但先试试直接传 CPU tensor（很多函数内部会自动分 batch 搬运）：
try:
    # 移除这里的 .cuda()，让函数内部去处理设备分配
    depths_uni, seq_flows_occs_fwd, seq_flows_occs_bwd = get_depth_flow(imgs_tensor, model)
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("⚠️ 全量处理依然 OOM，正在尝试分段处理...")
        # 如果还是崩，这里需要手动分 batch，比如每次只传 10 帧
        # 但通常 AnyCam 的 get_depth_flow 应该有内部 batch 逻辑
        torch.cuda.empty_cache()
        # 这里建议检查一下你的 model 是否在 cuda 上
        model = model.to("cuda") 
        # 降级方案：减小 imgs 的数量，比如 imgs[::12]

100%|████████████████████████████████████████████████████████████████████████████████| 247/247 [01:52<00:00,  2.20it/s]


In [ ]:
# Get DepthCrafter depths

dc_pipe = get_depth_crafter()

depths_dc = depth_crafter_inference(dc_pipe, torch.tensor(np.array(imgs), device="cuda"))

E:\AnyCam\anycam\env\lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

E:\AnyCam\anycam\env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--tencent--DepthCrafter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.05G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

...

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

Elapsed time for denoising video: 1015901.375 ms


Elapsed time for denoising video: 24647.189453125 ms
Elapsed time for decoding video: 22097.677734375 ms


In [21]:
uncertainties.shape

torch.Size([30, 1, 192, 192])

In [22]:
#depths_dc_uni = depth_alignment(depths_uni, depths_dc, uncertainties)


# 既然跳过了 DepthCrafter 对齐，我们将 AnyCam 预测的深度赋值给可视化变量
# 确保你已经运行了处理 depths 的那一步（即从 extras_dict 提取深度的步骤）

depths_dc_uni = depths  # 直接用 AnyCam 自带的深度代替对齐后的深度
uncertainties_1 = uncertainties # 确保不确定性变量名也对应上

In [23]:
#max_depth_masks = (depths_uni < 1.5) & (depths_dc_uni < 2)
#max_depth_masks = (depths_uni < 5) & (depths_dc_uni < 5)
# 既然我们只有一份深度数据（depths），就只针对它进行过滤
# 1.5 或 5 是距离单位，通常 5 米以内的点会被保留
max_depth_masks = (depths < 5.0) 

# 如果你之前没有做物体追踪（track_masks），为了防止可视化报错，补一行：
track_masks = torch.ones_like(max_depth_masks, dtype=torch.bool)

print("✅ 深度遮罩已创建，将过滤掉过远处的噪点。")

✅ 深度遮罩已创建，将过滤掉过远处的噪点。


In [42]:
uncertainties_1 = torch.cat((uncertainties, uncertainties[-1:]), dim=0)
uncert_thresh = 0.01

track_masks = compute_track_masks(seq_flows_occs_fwd.cuda(), uncertainties_1.cuda(), max_track_len=16, min_track_len=2, uncertainty_thresh=uncert_thresh, plot_rerun=False, imgs=imgs)

track_masks = track_masks > .5

In [ ]:
rr.init("depth optimization rerun", recording_id=uuid.uuid4())
rr.connect()

rr.log("world", rr.Clear(recursive=True))
rr.log("log", rr.Clear(recursive=True))
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)
rr.log("world/scene", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

# Dynamic continuous reconstruction

visualize_video_rerun(
    best_trajectory,
    depths_dc_uni,
    imgs,
    proj,
    uncertainties=uncertainties_1,
    cam_every=18,
    depth_every=1,
    subsample_pts=5, # 5
    subsample_dynamic_pts=1, # 2
    uncertainty_thresh=uncert_thresh,
    track_masks=track_masks,
    max_depth_masks=max_depth_masks.to(track_masks.device),
    follow_cam="world",
    radii=2,
    filter_depth_threshold=0.05,
    step=0,
    static_points_accumulate=True,
    highlight_dynamic=0.01,
    image_plane_distance=0.1,
)

/tmp/ipykernel_2210566/2931543681.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  proj = torch.tensor(proj, device=device).float()
/tmp/ipykernel_2210566/2931543681.py:48: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  colors = torch.tensor(img.reshape(-1, 3)).cuda()


In [35]:
import cv2
import rerun as rr
import torch
import numpy as np

# ==========================================
# 1. 暴力初始化（忽略 gRPC 清理报错）
# ==========================================
try:
    # 如果 init 报错，我们强行 pass，不让它中断
    rr.init("AnyCam_Rescue", spawn=False)
except Exception:
    print("⚠️ 忽略了 gRPC 清理报错，尝试强制连接...")

# 手动连接到默认端口
rr.connect("127.0.0.1:9876")
print("✅ 强制连接指令已发出，请确保你现在去终端输入 rerun 并启动它")

# ==========================================
# 2. 定义渲染函数
# ==========================================
def final_rescue_render(trajectory, depths, imgs):
    num_frames = min(len(trajectory), len(depths), len(imgs))
    target_h, target_w = depths[0].shape[-2:]
    
    grid_x, grid_y = np.meshgrid(np.arange(target_w), np.arange(target_h))
    grid_x = grid_x.reshape(-1)
    grid_y = grid_y.reshape(-1)

    for i in range(num_frames):
        rr.set_time_sequence("step", i)
        
        # 缩放图片对齐维度
        img_resized = cv2.resize(imgs[i], (target_w, target_h))
        
        current_depth = depths[i].cpu().numpy().reshape(-1) if torch.is_tensor(depths) else depths[i].reshape(-1)
        current_color = img_resized.reshape(-1, 3)

        # 提取平移 (处理 3x4 矩阵)
        pos = trajectory[i]
        trans = pos[:3, 3] if pos.shape == (3, 4) else pos[:3]

        # 抽样渲染
        skip = 8 # 8GB 显存建议抽样大一点，更稳
        points = np.zeros((len(current_depth[::skip]), 3))
        points[:, 0] = grid_x[::skip] * 0.02
        points[:, 1] = grid_y[::skip] * 0.02
        points[:, 2] = current_depth[::skip]

        rr.log("world/points", rr.Points3D(points, colors=current_color[::skip]))
        rr.log("world/camera", rr.Transform3D(translation=trans))

    print("✨ 渲染指令已全部发出！")

# ==========================================
# 3. 立即执行渲染
# ==========================================
# 在执行前，请去终端输入一次 rerun 启动窗口
try:
    v_depths = final_depths if 'final_depths' in locals() else depths
    v_imgs = imgs[::6] 
    final_rescue_render(best_trajectory, v_depths, v_imgs)
except Exception as e:
    print(f"❌ 运行失败: {e}")

⚠️ 忽略了 gRPC 清理报错，尝试强制连接...


AttributeError: module 'rerun' has no attribute 'connect'

In [ ]:
rr.init("depth optimization rerun", recording_id=uuid.uuid4())
rr.connect()

rr.log("world", rr.Clear(recursive=True))
rr.log("log", rr.Clear(recursive=True))
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)
rr.log("world/scene", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

# Dynamic discrete reconstruction

visualize_video_rerun(
    best_trajectory,
    depths_dc_uni,
    imgs,
    proj,
    uncertainties=uncertainties_1,
    cam_every=18,
    depth_every=18,
    subsample_pts=5, # 5
    subsample_dynamic_pts=1, # 2
    uncertainty_thresh=uncert_thresh,
    max_depth_masks=max_depth_masks.to(track_masks.device),
    follow_cam="world",
    radii=2,
    filter_depth_threshold=0.05,
    step=0,
    static_points_accumulate=True,
    highlight_dynamic=0.01,
    image_plane_distance=0.1,
)

/tmp/ipykernel_2210566/2931543681.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  proj = torch.tensor(proj, device=device).float()
/tmp/ipykernel_2210566/2931543681.py:48: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  colors = torch.tensor(img.reshape(-1, 3)).cuda()


In [130]:
# Export to input images, depth, flow, and uncertainties to video and frames

import cv2
from moviepy import VideoClip

def save_to_video_and_frames(frames, out_path, fps=30):
    out_path.mkdir(exist_ok=True, parents=True)
    frames_dir = out_path / "frames"
    frames_dir.mkdir(exist_ok=True, parents=True)

    for i, frame in enumerate(frames):
        cv2.imwrite(str(frames_dir / f"{i:04d}.png"), cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    clip = VideoClip(lambda t: frames[int(t * fps)], duration=len(frames) / fps)
    clip.write_videofile(str(out_path / "video.mp4"), codec="libx264", fps=fps)

def save_all(out_path, imgs, depths, uncertainties, flows_occs_fwd, fps=30):
    save_to_video_and_frames([(img * 255).astype(np.uint8) for img in imgs], out_path / "imgs", fps)

    depths_c = [(color_tensor(((1 / (depths[i, 0] * 10)) * 5).clamp(0, 1), cmap="plasma", norm=False) * 255).numpy().astype(np.uint8) for i in range(len(depths))]
    save_to_video_and_frames(depths_c, out_path / "depths", fps)

    uncertainties_c = [(color_tensor((uncertainties[i, 0] / 0.08).clamp(0, 1), cmap="plasma", norm=False) * 255).numpy().astype(np.uint8) for i in range(len(uncertainties))]
    save_to_video_and_frames(uncertainties_c, out_path / "uncertainties", fps)

    flows = flow_to_image(flows_occs_fwd[:, :2].cpu()).permute(0, 2, 3, 1).numpy()
    save_to_video_and_frames(flows, out_path / "flows_occs_fwd", fps)

save_all(Path("media/anycam/4D/running_dog_pexels"), imgs, depths_dc_uni, uncertainties, seq_flows_occs_fwd, fps=15)

MoviePy - Building video media/anycam/4D/running_dog_pexels/imgs/video.mp4.
MoviePy - Writing video media/anycam/4D/running_dog_pexels/imgs/video.mp4



MoviePy - Done !
MoviePy - video ready media/anycam/4D/running_dog_pexels/imgs/video.mp4
MoviePy - Building video media/anycam/4D/running_dog_pexels/depths/video.mp4.
MoviePy - Writing video media/anycam/4D/running_dog_pexels/depths/video.mp4



MoviePy - Done !
MoviePy - video ready media/anycam/4D/running_dog_pexels/depths/video.mp4
MoviePy - Building video media/anycam/4D/running_dog_pexels/uncertainties/video.mp4.
MoviePy - Writing video media/anycam/4D/running_dog_pexels/uncertainties/video.mp4



MoviePy - Done !
MoviePy - video ready media/anycam/4D/running_dog_pexels/uncertainties/video.mp4
MoviePy - Building video media/anycam/4D/running_dog_pexels/flows_occs_fwd/video.mp4.
MoviePy - Writing video media/anycam/4D/running_dog_pexels/flows_occs_fwd/video.mp4



MoviePy - Done !
MoviePy - video ready media/anycam/4D/running_dog_pexels/flows_occs_fwd/video.mp4


In [1]:
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
import polyscope as ps
import polyscope.imgui as imgui
import time
is_playing = False
is_reversed = True  # 新增：是否反向播放
# --- 配置 ---
VIDEO_PATH = "E:/AnyCam/anycam/video4.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project4")

# --- 状态变量 ---
current_frame_idx = 0
is_playing = False
last_update_time = 0
play_speed = 0.05  # 帧间隔时间（秒），调小这个值会播得更快
WINDOW_SIZE = 30

def lift_image_for_polyscope(img, depth, pose, focal=180.0):
    device = depth.device
    h, w = depth.shape
    
    img = torch.tensor(img).to(device).float() if not isinstance(img, torch.Tensor) else img.float()
    
    img_tmp = img.permute(2, 0, 1).unsqueeze(0)
    img_resized = F.interpolate(img_tmp, size=(h, w), mode='bilinear', align_corners=False)
    img = img_resized.squeeze(0).permute(1, 2, 0)

    i, j = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
    dirs = torch.stack([(j - w * .5) / focal, (i - h * .5) / focal, torch.ones_like(i)], -1).to(device)
    pts_cam = dirs * depth.float().unsqueeze(-1)
    
    pts_world = (pose.float()[:3, :3] @ pts_cam.reshape(-1, 3).T).T + pose.float()[:3, 3]
    return pts_world, img.reshape(-1, 3)

def update_view():
    global current_frame_idx, WINDOW_SIZE
    start_idx = max(0, current_frame_idx - WINDOW_SIZE)
    end_idx = min(len(all_p_polyscope), current_frame_idx + 1)
    
    if start_idx < end_idx:
        cp = np.concatenate(all_p_polyscope[start_idx:end_idx], axis=0)
        cc = np.concatenate(all_c_polyscope[start_idx:end_idx], axis=0)
        
        ps_pc = ps.register_point_cloud("4D_Cloud", cp)
        ps_pc.add_color_quantity("RGB", cc, enabled=True)
        ps_pc.set_radius(0.002, relative=False)

def callback():
    global current_frame_idx, WINDOW_SIZE, is_playing, last_update_time, play_speed, is_reversed
    
    # 1. 播放控制 UI
    imgui.Text("Playback Controls")
    _, is_playing = imgui.Checkbox("Auto Play", is_playing)
    imgui.SameLine() # 让复选框在一行显示
    _, is_reversed = imgui.Checkbox("Reverse", is_reversed) # 反向播放开关
    
    changed_frame, new_idx = imgui.SliderInt("Frame", current_frame_idx, 0, len(all_p_polyscope) - 1)
    if changed_frame:
        current_frame_idx = new_idx
        update_view()

    changed_win, new_win = imgui.SliderInt("Window Size", WINDOW_SIZE, 1, 100)
    if changed_win:
        WINDOW_SIZE = new_win
        update_view()

    # 2. 自动播放逻辑 (支持双向)
    if is_playing:
        current_time = time.time()
        if current_time - last_update_time > play_speed:
            if is_reversed:
                # --- 从后往前播放逻辑 ---
                current_frame_idx = (current_frame_idx - 1)
                if current_frame_idx < 0:
                    current_frame_idx = len(all_p_polyscope) - 1
            else:
                # --- 从前入后播放逻辑 ---
                current_frame_idx = (current_frame_idx + 1) % len(all_p_polyscope)
            
            update_view()
            last_update_time = current_time

# --- 主程序 ---
if __name__ == "__main__":
    cap = cv2.VideoCapture(VIDEO_PATH)
    raw_imgs = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        raw_imgs.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0)
    cap.release()

    depths_data = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda()
    traj_data = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda()

    min_len = min(len(raw_imgs), depths_data.shape[0], traj_data.shape[0])
    all_p_polyscope, all_c_polyscope = [], []
    
    print("预计算点云中...")
    for i in tqdm(range(min_len)):
        pts, colors = lift_image_for_polyscope(raw_imgs[i], depths_data[i, 0], traj_data[i])
        p_cpu = pts.cpu().numpy()[::2] # 预览用 step=2
        # 修正镜像映射
        p_fixed = np.stack([p_cpu[:, 0], -p_cpu[:, 2], p_cpu[:, 1]], axis=-1)
        all_p_polyscope.append(p_fixed)
        all_c_polyscope.append(colors.cpu().numpy()[::2])

    ps.init()
    ps.set_ground_plane_mode("none")
    update_view()
    ps.set_user_callback(callback)
    ps.show()

预计算点云中...


100%|█████████████████████████████████████████████████████████████████████████████████| 55/55 [00:00<00:00, 108.13it/s]


In [ ]:
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
import polyscope as ps
import polyscope.imgui as imgui
import time

# --- 配置 ---
VIDEO_PATH = "E:/AnyCam/anycam/video4.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project4")

# --- 状态变量 ---
current_frame_idx = 0
is_playing = False
is_reversed = False 
last_update_time = time.time()
play_speed = 0.08  
WINDOW_SIZE = 5 # 减小窗口，能减少重影
TRAJECTORY_SCALE = 5.0 # 轨迹缩放

def get_synced_frames(video_path, target_count):
    """【核心修复】确保原视频画面与 NPY 数据时间戳对齐"""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # 按照比例均匀抽取帧，而不是取前 target_count 帧
    indices = np.linspace(0, total_frames - 1, target_count).astype(int)
    
    frames = []
    print(f"同步原视频帧 (从 {total_frames} 帧中抽取 {target_count} 帧)...")
    for idx in tqdm(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            # AnyCam 默认可能使用的是 RGB，OpenCV 是 BGR
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0)
    cap.release()
    return frames

def lift_image_refined(img, depth, pose, focal=None, traj_scale=1.0):
    """精细化 3D 提升：处理对齐和拖尾"""
    device = depth.device
    h, w = depth.shape
    
    # 如果没设焦距，根据图像尺寸估算一个通用值 (通常 w * 0.8 到 1.2 之间)
    if focal is None:
        focal = w * 1.2 

    if not isinstance(img, torch.Tensor):
        img = torch.from_numpy(img).to(device).float()
    
    # 画面对齐深度图尺寸
    img_tmp = img.permute(2, 0, 1).unsqueeze(0)
    img_resized = F.interpolate(img_tmp, size=(h, w), mode='bilinear', align_corners=False)
    colors = img_resized.squeeze(0).permute(1, 2, 0).reshape(-1, 3)

    # 坐标生成
    i, j = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
    # 使用图像中心作为主点
    dirs = torch.stack([(j - w * 0.5) / focal, (i - h * 0.5) / focal, torch.ones_like(i)], -1).to(device)
    
    pts_cam = dirs * depth.float().unsqueeze(-1)
    
    # 变换位姿
    R = pose.float()[:3, :3]
    t = pose.float()[:3, 3] * traj_scale 
    
    pts_world = (R @ pts_cam.reshape(-1, 3).T).T + t
    
    # 【边缘剔除】尝试解决“桌子鼓包”：如果深度变化率太大，可能是边缘拖尾，不渲染（可选）
    # 这里简单处理：转为 numpy
    pts_np = pts_world.cpu().numpy()
    colors_np = colors.cpu().numpy()
    
    return pts_np, colors_np

def update_view():
    global current_frame_idx, WINDOW_SIZE, all_p_polyscope, all_c_polyscope, all_poses, TRAJECTORY_SCALE
    
    # 动态显示一个窗口内的点云累积
    start_idx = max(0, current_frame_idx - WINDOW_SIZE)
    end_idx = min(len(all_p_polyscope), current_frame_idx + 1)
    
    if start_idx < end_idx:
        cp = np.concatenate(all_p_polyscope[start_idx:end_idx], axis=0)
        cc = np.concatenate(all_c_polyscope[start_idx:end_idx], axis=0)
        
        ps_pc = ps.register_point_cloud("Scene_Cloud", cp)
        ps_pc.add_color_quantity("RGB", cc, enabled=True)
        ps_pc.set_radius(0.0015, relative=False)

        # 渲染轨迹
        path_nodes = []
        for i in range(max(0, current_frame_idx - 50), current_frame_idx + 1):
            t = all_poses[i][:3, 3] * TRAJECTORY_SCALE
            path_nodes.append([t[0], -t[2], t[1]])
        
        if len(path_nodes) > 1:
            path_nodes = np.array(path_nodes)
            path_edges = np.arange(len(path_nodes)-1)[:, None] + [0, 1]
            ps.register_curve_network("Camera_Path", path_nodes, path_edges).set_color((0, 0.5, 1))

def callback():
    global current_frame_idx, is_playing, last_update_time, play_speed
    imgui.Text(f"Current Frame: {current_frame_idx}")
    _, is_playing = imgui.Checkbox("Play", is_playing)
    
    changed, new_idx = imgui.SliderInt("Go to", current_frame_idx, 0, len(all_p_polyscope)-1)
    if changed:
        current_frame_idx = new_idx
        update_view()

    if is_playing and (time.time() - last_update_time > play_speed):
        current_frame_idx = (current_frame_idx + 1) % len(all_p_polyscope)
        update_view()
        last_update_time = time.time()

if __name__ == "__main__":
    # 1. 加载数据
    depths_data = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda()
    traj_data = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda()
    num_results = depths_data.shape[0]

    # 2. 【核心修复】获取同步帧
    raw_imgs = get_synced_frames(VIDEO_PATH, num_results)

    all_p_polyscope, all_c_polyscope, all_poses = [], [], []
    
    print("预计算对齐点云...")
    for i in tqdm(range(num_results)):
        # 传入 None 让函数自动计算 focal
        pts, colors = lift_image_refined(raw_imgs[i], depths_data[i, 0], traj_data[i], traj_scale=TRAJECTORY_SCALE)
        
        # 降采样预览
        all_p_polyscope.append(np.stack([pts[::2, 0], -pts[::2, 2], pts[::2, 1]], axis=-1))
        all_c_polyscope.append(colors[::2])
        all_poses.append(traj_data[i].cpu().numpy())

    ps.init()
    ps.set_ground_plane_mode("none")
    update_view()
    ps.set_user_callback(callback)
    ps.show()

同步原视频帧 (从 174 帧中抽取 58 帧)...


100%|██████████████████████████████████████████████████████████████████████████████████| 58/58 [00:02<00:00, 21.35it/s]


预计算对齐点云...


100%|█████████████████████████████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 126.81it/s]

In [1]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video2.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project2")

# 参数设置
UNCERTAINTY_THRESH = 0.15      # 不确定度阈值
FILTER_DEPTH_THRESHOLD = 0.1   # 深度梯度阈值 (消除运动残影核心)
SUBSAMPLE = 4                  # 点云降采样，防止太卡
# ==========================================

def filter_depth_gradient(depth, threshold=0.1):
    """
    通过计算深度的空间梯度，剔除深度突变的边缘点，彻底消除汽车残影
    """
    if depth.dim() == 2:
        depth = depth.clone().unsqueeze(0)
    else:
        depth = depth.clone()
        
    median = torch.median(depth)
    depth_grad = torch.stack(torch.gradient(depth, dim=(-2, -1))).norm(dim=0)
    mask = depth_grad < median * threshold
    return mask.squeeze()

def main():
    print("📂 正在加载 AnyCam 预测的深度和轨迹数据...")
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda()
    uncertainties = torch.from_numpy(np.load(RESULTS_DIR / "uncertainties.npy")).cuda()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    v_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx_map = np.linspace(0, v_total - 1, num_frames).astype(int)
    
    # 获取图像尺寸和估算焦距
    ret, first_frame = cap.read()
    if not ret:
        print("❌ 无法读取视频！请检查 VIDEO_PATH")
        return
    h, w = first_frame.shape[:2]
    focal_length = w * 0.9 

    # 1. 初始化 Rerun (spawn=True 会自动弹出渲染器窗口)
    rr.init("AnyCam_Scene_Reconstruction", spawn=True)
    
    # 强制设置世界坐标系为“右手 Y轴朝下”，这能彻底解决任何颠倒问题
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    trajectory_points = []
    
    print("🚀 开始向 Rerun 引擎推送 3D 时序点云...")
    for i in tqdm(range(num_frames)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx_map[i])
        ret, frame = cap.read()
        if not ret: break
        
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        depth = depths[i, 0]
        unc = uncertainties[i, 0]
        pose = trajectories[i]

        # --- 核心修复：兼容新版 Rerun API ---
        # 使用 set_time 替代被弃用的 set_time_sequence
        rr.set_time("step", sequence=i)

        # --- 核心过滤器：消除残影 ---
        grad_mask = filter_depth_gradient(depth, threshold=FILTER_DEPTH_THRESHOLD)
        unc_mask = unc < UNCERTAINTY_THRESH
        valid_mask = grad_mask & unc_mask & (depth > 0.1) & (depth < 10.0)
        
        # 提取 3D 坐标
        v, u = torch.where(valid_mask)
        z = depth[v, u]
        x = (u - w * 0.5) * z / focal_length
        y = (v - h * 0.5) * z / focal_length
        pts_cam = torch.stack([x, y, z], dim=-1)
        colors = torch.from_numpy(img_rgb).cuda()[v, u]
        
        # 降采样提速
        pts_cam = pts_cam[::SUBSAMPLE]
        colors = colors[::SUBSAMPLE]
        
        # 通过姿态矩阵(Extrinsic)转换到世界坐标系
        # --- 修复：确保所有张量类型一致 (均为 float32) ---
        R = pose[:3, :3].float()  # 强制转为 float32
        t = pose[:3, 3].float()   # 强制转为 float32
        pts_cam = pts_cam.float() # 确保点云也是 float32
        
        # 执行坐标变换
        pts_world = (R @ pts_cam.T).T + t
        
        # 2. 推送场景点云 (static=True 表示点云会不断累加在场景中)
        if len(pts_world) > 0:
            rr.log(
                f"world/scene/points_{i:03d}", 
                rr.Points3D(pts_world.cpu().numpy(), colors=colors.cpu().numpy(), radii=0.01), 
                static=True 
            )

        # 3. 推送当前活跃的相机组件
        # 推送相机的内外参
        rr.log("world/active_cam/image", rr.Pinhole(resolution=[w, h], focal_length=focal_length))
        # 推送相机的画面 
        rr.log("world/active_cam/image", rr.Image(img_rgb))
        # 推送相机的 3D 姿态
        rr.log("world/active_cam", rr.Transform3D(translation=t.cpu().numpy(), mat3x3=R.cpu().numpy()))

        # 4. 推送动态的蓝色相机轨迹
        trajectory_points.append(t.cpu().numpy().tolist())
        if len(trajectory_points) > 1:
            rr.log("world/scene/cam_traj", rr.LineStrips3D([trajectory_points], colors=[(0, 150, 255)]))

    print("✅ 推送完成！尽情在弹出的 Rerun 交互面板中探索 3D 场景吧。")

if __name__ == "__main__":
    main()

📂 正在加载 AnyCam 预测的深度和轨迹数据...
🚀 开始向 Rerun 引擎推送 3D 时序点云...


100%|██████████████████████████████████████████████████████████████████████████████████| 75/75 [00:03<00:00, 20.31it/s]

✅ 推送完成！尽情在弹出的 Rerun 交互面板中探索 3D 场景吧。


In [6]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video2.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project2")

STRETCH_Z = 80.0               # 调节这个值来控制路面“飞过来”的速度
FOV_SCALE = 0.45               
POINT_RADII = 0.0015            
UNCERTAINTY_THRESH = 0.9       
SAMPLE_STEP = 1
# ==========================================

def main():
    print("🚀 正在切换为【相机中心视角】模式...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_FPV_Fixed", spawn=True)
    # 强制固定世界坐标系
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    # 第一帧作为基准
    first_pose_inv = torch.inverse(trajectories[0])
    
    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算当前帧相对于第一帧的逆变换
        # 这一步是关键：我们不让相机往前走，而是让世界往后退
        current_pose = trajectories[i]
        rel_pose = first_pose_inv @ current_pose
        R = rel_pose[:3, :3]
        t_vec = rel_pose[:3, 3] * STRETCH_Z

        # 2. 生成点云
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        # 采样步长改为 SAMPLE_STEP
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        colors = torch.from_numpy(img_resized).cuda()[v_idx, u_idx][::SAMPLE_STEP]

        # 核心修改：让点云位置更精确
        pts_relative = (R @ pts_cam.T).T + t_vec

        # 推送 Rerun
        rr.log(
            "world/scene", 
            rr.Points3D(pts_relative.cpu().numpy(), colors=colors.cpu().numpy(), radii=POINT_RADII)
        )

        # 相机固定在原点
        rr.log("world/camera", rr.Transform3D(translation=[0, 0, 0], mat3x3=np.eye(3)))
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))

    print("✅ 已切换为相机固定模式。")

if __name__ == "__main__":
    main()

🚀 正在切换为【相机中心视角】模式...


100%|██████████████████████████████████████████████████████████████████████████████████| 75/75 [00:03<00:00, 21.09it/s]

✅ 已切换为相机固定模式。


In [1]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results")

STRETCH_Z = 80.0               # 增加拉伸感，让路面飞得更顺滑
FOV_SCALE = 0.45               
POINT_RADII = 0.0015            
UNCERTAINTY_THRESH = 0.9       
SAMPLE_STEP = 1                 # 设为2平衡密度与性能
# ==========================================

def main():
    print("🚀 启动相机固定模式 + 动态轨迹跟踪...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_FPV_Fixed_With_Track", spawn=True)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    # 第一帧作为基准
    first_pose_inv = torch.inverse(trajectories[0])
    
    # 🌟 用于存储轨迹历史的列表
    trajectory_history = []
    
    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算相对位移
        current_pose = trajectories[i]
        rel_pose = first_pose_inv @ current_pose
        R = rel_pose[:3, :3]
        t_vec = rel_pose[:3, 3] * STRETCH_Z
        
        # 🌟 将当前位移存入历史
        trajectory_history.append(t_vec.cpu().numpy())

        # 2. 生成点云
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        colors = torch.from_numpy(img_resized).cuda()[v_idx, u_idx][::SAMPLE_STEP]

        # 核心：点云随世界后退
        pts_relative = (R @ pts_cam.T).T + t_vec

        # 3. 推送 Rerun
        # --- 点云 ---
        rr.log(
            "world/scene", 
            rr.Points3D(pts_relative.cpu().numpy(), colors=colors.cpu().numpy(), radii=POINT_RADII)
        )

        # --- 🌟 动态轨迹线 ---
        # 轨迹线也要跟着 R 和 t_vec 移动，才能保证它始终连在相机后面
        if len(trajectory_history) > 1:
            # 将历史轨迹点转换为 numpy 数组
            path_points = np.array(trajectory_history)
            # 这里的逻辑是：轨迹线记录的是相机走过的路，
            # 在相机固定模式下，历史轨迹点也会随着场景一起向后位移。
            rr.log(
                "world/scene/trajectory",
                rr.LineStrips3D([path_points], colors=[[255, 0, 0]], radii=0.02)
            )

        # --- 相机固定在原点 ---
        rr.log("world/camera", rr.Transform3D(translation=[0, 0, 0], mat3x3=np.eye(3)))
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))

    print("✅ 相机中心模式 + 动态轨迹渲染完成。")

if __name__ == "__main__":
    main()

🚀 启动相机固定模式 + 动态轨迹跟踪...


100%|██████████████████████████████████████████████████████████████████████████████████| 42/42 [00:02<00:00, 15.90it/s]

✅ 相机中心模式 + 动态轨迹渲染完成。


In [2]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video5.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project5")

STRETCH_Z = 80.0               # 增加拉伸感，让路面飞得更顺滑
FOV_SCALE = 0.45               
POINT_RADII = 0.0015            
UNCERTAINTY_THRESH = 0.9       
SAMPLE_STEP = 2                 # 平衡密度与性能，设为2
# ==========================================

def main():
    print("🚀 启动相机固定模式 + 动态轨迹 + 视锥体显示...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_FPV_Fixed_With_Pyramid", spawn=True)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    # 第一帧作为基准
    first_pose_inv = torch.inverse(trajectories[0])
    
    # 用于存储轨迹历史的列表
    trajectory_history = []
    
    # 🌟 [关键修复]: 定义一个静态的三棱锥几何体 (以原点为顶点，朝向Z轴)
    # 这只是一个简易的框架，代表相机的视锥
    pyramid_scale = 0.5 # 视锥体的大小
    aspect = w_img / h_img
    
    # 5个顶点: 顶点(0,0,0) + 近平面的4个角
    pyramid_verts = np.array([
        [0, 0, 0], # 顶点
        [-pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2], # 左上
        [pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2],  # 右上
        [pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],   # 右下
        [-pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],  # 左下
    ])
    
    # 连线索引 (将顶点连接成框架)
    pyramid_strips = np.array([
        [0, 1, 2, 0, 3, 2, 3, 4, 0, 4, 1] # 连成一个视锥体框架
    ])

    print(f"📊 正在处理 {num_frames} 帧视频...")
    
    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算相对位移
        current_pose = trajectories[i]
        rel_pose = first_pose_inv @ current_pose
        R = rel_pose[:3, :3].cpu().numpy() # 拿到相机的旋转逆变换
        t_vec = rel_pose[:3, 3] * STRETCH_Z
        
        # 将当前位移存入历史
        trajectory_history.append(t_vec.cpu().numpy())

        # 2. 生成点云
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        # 优化：降采样以提高性能，设为SAMPLE_STEP
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        colors = torch.from_numpy(img_resized).cuda()[v_idx, u_idx][::SAMPLE_STEP]

        # 核心：点云随世界后退 (应用相机的相对旋转和位移)
        pts_relative = (rel_pose[:3, :3] @ pts_cam.T).T + t_vec

        # 3. 推送 Rerun
        # --- 点云 ---
        if len(pts_relative) > 0:
            rr.log(
                "world/scene", 
                rr.Points3D(pts_relative.cpu().numpy(), colors=colors.cpu().numpy(), radii=POINT_RADII)
            )

        # --- 🌟 动态视锥体 (三棱锥) ---
        # 核心逻辑：虽然相机t=[0,0,0]，但我们要把AnyCam预测的相机旋转应用到这个模型上
        # 这样它看起来就在原点摆动
        # 使用 LineStrips3D 推送旋转后的几何体
        
        # 应用旋转 R 到三棱锥顶点
        rotated_pyramid_verts = (R @ pyramid_verts.T).T
        
        # 推送视锥体框架 (颜色设为青色以区别轨迹)
        rr.log(
            "world/camera/frustum",
            rr.LineStrips3D(rotated_pyramid_verts[pyramid_strips], colors=[[0, 255, 255]], radii=0.01)
        )

        # --- 动态轨迹线 ---
        if len(trajectory_history) > 1:
            # 将历史轨迹点转换为 numpy 数组
            path_points = np.array(trajectory_history)
            # 在相机固定模式下，历史轨迹点也会随着场景一起向后位移。
            rr.log(
                "world/scene/trajectory",
                rr.LineStrips3D([path_points], colors=[[255, 0, 0]], radii=0.015)
            )

        # --- 🌟 修改相机的 Transform3D 日志 ---
        # 这一步是让 Rerun 知道这个 Transform 是动态更新旋转的
        rr.log("world/camera", rr.Transform3D(translation=[0, 0, 0], mat3x3=R))
        
        # 相机内参和图像保持不变
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))

    print("✅ 视锥体+动态轨迹跟踪渲染完成。")

if __name__ == "__main__":
    main()

🚀 启动相机固定模式 + 动态轨迹 + 视锥体显示...
📊 正在处理 68 帧视频...


100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [00:06<00:00, 10.61it/s]

✅ 视锥体+动态轨迹跟踪渲染完成。


In [9]:
import cv2
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video3.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project3")
OUTPUT_VIDEO = "uncertainty_fixed_scale.mp4"

# 🎨 颜色映射配置
# 如果画面还是太蓝，请将 VMAX 调小（例如 0.02）；如果太红，请调大。
VMIN = 0.0    
VMAX = 0.2   # 固定最大值：在此数值以上的点都会显示为深红色
FPS = 15
# ==========================================

def main():
    print("🚀 正在分析数据特征...")
    
    # 1. 加载数据 [N, 1, H, W]
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    if not u_path.exists():
        print(f"❌ 错误：在 {RESULTS_DIR} 下找不到文件。")
        return
        
    uncertainties = np.load(u_path)
    num_frames = uncertainties.shape[0]

    # 打印统计信息，帮你决定 VMAX
    u_min, u_max, u_mean = uncertainties.min(), uncertainties.max(), uncertainties.mean()
    print(f"📈 全局统计量: Min={u_min:.5f}, Max={u_max:.5f}, Mean={u_mean:.5f}")
    print(f"💡 当前设定 VMAX={VMAX}。建议将 VMAX 设为 Mean 的 2-3 倍左右。")

    # 2. 视频读取器
    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h, w = frame.shape[:2]
    
    # 3. 视频写入器 (左右拼接)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, FPS, (w * 2, h))

    # 4. 逐帧处理
    frame_indices = np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames).astype(int)

    print(f"🎬 开始同步渲染 (固定量程: {VMIN} - {VMAX})...")
    for i in tqdm(range(num_frames)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_indices[i])
        ret, frame = cap.read()
        if not ret: break

        # 获取原始不确定度数据
        unc_map = uncertainties[i, 0] 
        
        # --- 核心：固定范围归一化 ---
        # 强制将 [VMIN, VMAX] 映射到 [0, 1]，超过范围的截断
        unc_clipped = np.clip((unc_map - VMIN) / (VMAX - VMIN), 0, 1)
        unc_norm = (unc_clipped * 255).astype(np.uint8)
        
        # 缩放并上色
        unc_resized = cv2.resize(unc_norm, (w, h))
        unc_color = cv2.applyColorMap(unc_resized, cv2.COLORMAP_JET)

        # 5. UI 标注
        # 原图左侧标注
        cv2.rectangle(frame, (10, 10), (320, 70), (0,0,0), -1)
        cv2.putText(frame, "Original Video", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # 热力图右侧标注
        cv2.rectangle(unc_color, (10, 10), (500, 110), (0,0,0), -1)
        cv2.putText(unc_color, f"Uncertainty (Scale: {VMIN}-{VMAX})", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(unc_color, f"Red = High Uncertainty (> {VMAX})", (20, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # 左右拼接
        combined = np.hstack((frame, unc_color))
        out.write(combined)

    cap.release()
    out.release()
    print(f"✅ 渲染完成！视频已保存至: {OUTPUT_VIDEO}")

if __name__ == "__main__":
    main()

🚀 正在分析数据特征...
📈 全局统计量: Min=0.00000, Max=0.30444, Mean=0.00536
💡 当前设定 VMAX=0.2。建议将 VMAX 设为 Mean 的 2-3 倍左右。
🎬 开始同步渲染 (固定量程: 0.0 - 0.2)...


100%|██████████████████████████████████████████████████████████████████████████████████| 57/57 [00:04<00:00, 13.99it/s]

✅ 渲染完成！视频已保存至: uncertainty_fixed_scale.mp4


In [1]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video5.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project5")

STRETCH_Z = 80.0               # 轨迹的纵向拉伸
FOV_SCALE = 0.45               
POINT_RADII = 0.0015            
UNCERTAINTY_THRESH = 0.9       
SAMPLE_STEP = 4                 # 相机移动模式建议设为4，平衡性能
# ==========================================

def main():
    print("🚀 启动相机运动模式 (World-Space)...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_Moving_Camera", spawn=True)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    # 第一帧作为基准点
    first_pose = trajectories[0].clone()
    
    # 定义视锥体三棱锥模型 (本地坐标系)
    pyr_s = 0.4 
    aspect = w_img / h_img
    pyramid_verts = np.array([
        [0, 0, 0], 
        [-pyr_s*aspect, -pyr_s, pyr_s*2], [pyr_s*aspect, -pyr_s, pyr_s*2],
        [pyr_s*aspect, pyr_s, pyr_s*2], [-pyr_s*aspect, pyr_s, pyr_s*2]
    ])
    pyramid_strips = np.array([[0, 1, 2, 0, 3, 2, 3, 4, 0, 4, 1]])

    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算相机在世界坐标系中的位姿
        current_pose = trajectories[i]
        R = current_pose[:3, :3].cpu().numpy()
        # 计算相对于起点的位移并拉伸
        t_world = (current_pose[:3, 3] - first_pose[:3, 3]) * STRETCH_Z
        t_world_np = t_world.cpu().numpy()

        # 2. 生成当前帧的点云并变换到世界系
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        colors = torch.from_numpy(img_resized).cuda()[v_idx, u_idx][::SAMPLE_STEP]

        # 将点云从相机系转到世界系
        pts_world = (torch.from_numpy(R).cuda() @ pts_cam.T).T + t_world

        # 3. 推送 Rerun
        # --- 运动相机 (三棱锥) ---
        # 我们把三棱锥放在计算出的世界位置 t_world 上
        rr.log("world/camera", rr.Transform3D(translation=t_world_np, mat3x3=R))
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))
        
        # 视锥体线框
        rr.log("world/camera/frustum", rr.LineStrips3D(pyramid_verts[pyramid_strips], colors=[[0, 255, 255]], radii=0.01))

        # --- 世界点云 ---
        # 注意：这里我们每一帧都推送一个新的路径点，这样点云会留在原地
        # 为了避免 Rerun 内存过载，我们给每一帧的点云一个唯一的路径
        rr.log(
            f"world/points/frame_{i}", 
            rr.Points3D(pts_world.cpu().numpy(), colors=colors.cpu().numpy(), radii=POINT_RADII)
        )

        # --- 轨迹线 ---
        rr.log("world/track", rr.Points3D([t_world_np], colors=[[255, 0, 0]], radii=0.02))

    print("✅ 相机运动模式渲染完成。")

if __name__ == "__main__":
    main()

🚀 启动相机运动模式 (World-Space)...


100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [00:06<00:00, 10.60it/s]

✅ 相机运动模式渲染完成。


In [3]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video6.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project6")

STRETCH_Z = 80.0               
FOV_SCALE = 0.45               
# 🌟 修改1：增加点的渲染半径，让点云看起来更密实，像一个曲面
POINT_RADII = 0.004            
# 🌟 修改2：放宽不确定性阈值 (从0.9提高到1.5或2.0)，保留更多的点
UNCERTAINTY_THRESH = 0.1       
# 🌟 修改3：取消降采样，使用全分辨率点云
SAMPLE_STEP = 1                
# ==========================================

def main():
    print("🚀 启动相机固定模式 + 动态轨迹 + 视锥体显示...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_FPV_Fixed_With_Pyramid", spawn=True)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    first_pose_inv = torch.inverse(trajectories[0])
    trajectory_history = []
    
    pyramid_scale = 0.5 
    aspect = w_img / h_img
    
    pyramid_verts = np.array([
        [0, 0, 0], 
        [-pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2], 
        [pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2],  
        [pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],   
        [-pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],  
    ])
    
    pyramid_strips = np.array([
        [0, 1, 2, 0, 3, 2, 3, 4, 0, 4, 1] 
    ])

    print(f"📊 正在处理 {num_frames} 帧视频...")
    
    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        
        # 获取RGB图像 (类型为 uint8)
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算相对位移
        current_pose = trajectories[i]
        rel_pose = first_pose_inv @ current_pose
        R = rel_pose[:3, :3].cpu().numpy()
        t_vec = rel_pose[:3, 3] * STRETCH_Z
        
        trajectory_history.append(t_vec.cpu().numpy())

        # 2. 生成点云
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        
        # 生成 3D 坐标并降采样
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        # 🌟 修改4：优化颜色提取逻辑，直接在 Numpy 中操作，确保类型正确
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        # 将索引转移到 CPU numpy 中进行提取，防止 PyTorch/CUDA 切片造成的类型或连续性问题
        v_idx_np = v_idx.cpu().numpy()
        u_idx_np = u_idx.cpu().numpy()
        
        # 提取颜色，并强制转换为连续的 uint8 数组
        colors_np = img_resized[v_idx_np, u_idx_np][::SAMPLE_STEP]
        colors_np = np.ascontiguousarray(colors_np, dtype=np.uint8)

        # 核心：点云随世界后退
        pts_relative = (rel_pose[:3, :3] @ pts_cam.T).T + t_vec
        pts_relative_np = np.ascontiguousarray(pts_relative.cpu().numpy())

        # 3. 推送 Rerun
        if len(pts_relative_np) > 0:
            rr.log(
                "world/scene", 
                # 🌟 修改5：传入处理好的 numpy 颜色数组
                rr.Points3D(pts_relative_np, colors=colors_np, radii=POINT_RADII)
            )

        # 动态视锥体
        rotated_pyramid_verts = (R @ pyramid_verts.T).T
        rr.log(
            "world/camera/frustum",
            rr.LineStrips3D(rotated_pyramid_verts[pyramid_strips], colors=[[0, 255, 255]], radii=0.01)
        )

        # 动态轨迹线
        if len(trajectory_history) > 1:
            path_points = np.array(trajectory_history)
            rr.log(
                "world/scene/trajectory",
                rr.LineStrips3D([path_points], colors=[[255, 0, 0]], radii=0.015)
            )

        # 相机姿态与图像
        rr.log("world/camera", rr.Transform3D(translation=[0, 0, 0], mat3x3=R))
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))

    print("✅ 视锥体+动态轨迹跟踪渲染完成。")

if __name__ == "__main__":
    main()

🚀 启动相机固定模式 + 动态轨迹 + 视锥体显示...
📊 正在处理 69 帧视频...


100%|██████████████████████████████████████████████████████████████████████████████████| 69/69 [00:02<00:00, 30.80it/s]

✅ 视锥体+动态轨迹跟踪渲染完成。


In [3]:
import cv2
import numpy as np
import torch
import rerun as rr
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
VIDEO_PATH = "E:/AnyCam/anycam/video6.mp4"
RESULTS_DIR = Path("E:/AnyCam/anycam/my_results/project6")

STRETCH_Z = 80.0               
FOV_SCALE = 0.45               
POINT_RADII = 0.004            
UNCERTAINTY_THRESH = 0.05       
SAMPLE_STEP = 1                
# ==========================================

def main():
    print("🚀 启动相机跟随模式 + 动态轨迹 + 迷你视锥体...")
    
    depths = torch.from_numpy(np.load(RESULTS_DIR / "depths.npy")).cuda().float()
    u_path = RESULTS_DIR / "uncertainties.npy" if (RESULTS_DIR / "uncertainties.npy").exists() else RESULTS_DIR / "uncertainty.npy"
    uncertainties = torch.from_numpy(np.load(u_path)).cuda().float()
    trajectories = torch.from_numpy(np.load(RESULTS_DIR / "trajectory.npy")).cuda().float()
    num_frames = depths.shape[0]

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret: return
    h_img, w_img = frame.shape[:2]
    _, _, h_depth, w_depth = depths.shape
    scale_u, scale_v = w_img / w_depth, h_img / h_depth
    focal_length = w_img * FOV_SCALE 

    rr.init("AnyCam_Follow_Mode", spawn=True)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Y_DOWN, static=True)

    first_pose_inv = torch.inverse(trajectories[0])
    trajectory_history = []
    
    # 🌟 修改1：显著减小视锥体尺寸 (从 0.5 降到 0.1)，避免遮挡
    pyramid_scale = 0.1 
    aspect = w_img / h_img
    
    # 定义原始视锥体几何 (顶点在原点)
    pyramid_verts = np.array([
        [0, 0, 0], 
        [-pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2], 
        [pyramid_scale * aspect, -pyramid_scale, pyramid_scale * 2],  
        [pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],   
        [-pyramid_scale * aspect, pyramid_scale, pyramid_scale * 2],  
    ])
    
    pyramid_strips = np.array([
        [0, 1, 2, 0, 3, 2, 3, 4, 0, 4, 1] 
    ])

    print(f"📊 正在处理 {num_frames} 帧视频...")
    
    for i in tqdm(range(num_frames)):
        rr.set_time("step", sequence=i)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(np.linspace(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))-1, num_frames)[i]))
        ret, frame = cap.read()
        if not ret: break
        
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1. 计算当前位姿
        current_pose = trajectories[i]
        rel_pose = first_pose_inv @ current_pose
        R = rel_pose[:3, :3].cpu().numpy()
        t_vec = rel_pose[:3, 3] * STRETCH_Z
        t_vec_np = t_vec.cpu().numpy() # 转换为 numpy 方便计算
        
        trajectory_history.append(t_vec_np)

        # 2. 生成点云 (相对于当前相机位姿平移)
        depth = depths[i, 0]
        mask = (uncertainties[i, 0] < UNCERTAINTY_THRESH) & (depth > 0.1)
        v_idx, u_idx = torch.where(mask)
        z = depth[v_idx, u_idx]

        x = (u_idx.float() * scale_u - w_img * 0.5) * z / focal_length
        y = (v_idx.float() * scale_v - h_img * 0.5) * z / focal_length
        
        pts_cam = torch.stack([x, y, z], dim=-1)[::SAMPLE_STEP]
        
        img_resized = cv2.resize(img_rgb, (w_depth, h_depth))
        v_idx_np = v_idx.cpu().numpy()
        u_idx_np = u_idx.cpu().numpy()
        colors_np = np.ascontiguousarray(img_resized[v_idx_np, u_idx_np][::SAMPLE_STEP], dtype=np.uint8)

        # 点云位置：应用旋转并平移到 t_vec
        pts_relative = (rel_pose[:3, :3] @ pts_cam.T).T + t_vec
        pts_relative_np = np.ascontiguousarray(pts_relative.cpu().numpy())

        # 3. 推送 Rerun 数据
        # --- 点云 ---
        if len(pts_relative_np) > 0:
            rr.log("world/scene", rr.Points3D(pts_relative_np, colors=colors_np, radii=POINT_RADII))

        # --- 🌟 修改2：视锥体跟随红线移动 ---
        # 核心逻辑：旋转后的顶点坐标 + 当前平移向量 t_vec_np
        moved_pyramid_verts = (R @ pyramid_verts.T).T + t_vec_np
        
        rr.log(
            "world/camera/frustum",
            rr.LineStrips3D(moved_pyramid_verts[pyramid_strips], colors=[[0, 255, 255]], radii=0.008)
        )

        # --- 轨迹线 ---
        if len(trajectory_history) > 1:
            path_points = np.array(trajectory_history)
            rr.log("world/scene/trajectory", rr.LineStrips3D([path_points], colors=[[255, 0, 0]], radii=0.015))

        # --- 🌟 修改3：更新相机的 Transform 坐标 ---
        # translation 不再是 [0,0,0]，而是随轨迹移动的 t_vec_np
        rr.log("world/camera", rr.Transform3D(translation=t_vec_np, mat3x3=R))
        
        # 相机内参和预览图
        rr.log("world/camera/image", rr.Pinhole(resolution=[w_img, h_img], focal_length=focal_length))
        rr.log("world/camera/image", rr.Image(img_rgb))

    print("✅ 渲染完成：视锥体已缩小并跟随轨迹。")

if __name__ == "__main__":
    main()

🚀 启动相机跟随模式 + 动态轨迹 + 迷你视锥体...
📊 正在处理 69 帧视频...


100%|██████████████████████████████████████████████████████████████████████████████████| 69/69 [00:02<00:00, 28.85it/s]

✅ 渲染完成：视锥体已缩小并跟随轨迹。
